# L07 – Lösningar
Kräver `dlkit/train.py` i L07-versionen (med `patience`).

In [ ]:
import sys
sys.path.insert(0, "..")
import torch
import torch.nn as nn
from dlkit.seed import set_seed
from dlkit.data import fashion_mnist_loaders
from dlkit.train import fit

device = "cuda" if torch.cuda.is_available() else "cpu"
loss_fn = nn.CrossEntropyLoss()
train_loader, val_loader, _ = fashion_mnist_loaders(train_subset=2000)

def make_mlp(hidden=512):
    return nn.Sequential(nn.Flatten(), nn.Linear(784, hidden), nn.ReLU(),
                         nn.Linear(hidden, hidden), nn.ReLU(), nn.Linear(hidden, 10))

set_seed(42)
model = make_mlp().to(device)
h_base = fit(model, train_loader, val_loader, loss_fn,
             torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=40, verbose=False)

### Övning 1

In [ ]:
def my_dropout(x, p):
    mask = (torch.rand_like(x) >= p).float()   # 1 med sannolikheten 1-p
    return x * mask / (1 - p)

set_seed(0)
out = my_dropout(torch.ones(100_000), p=0.3)
assert torch.allclose(out[out > 0], torch.tensor(1 / 0.7))
assert abs((out == 0).float().mean().item() - 0.3) < 0.01
assert abs(out.mean().item() - 1.0) < 0.01
print("Rätt!")

### Övning 2

In [ ]:
set_seed(42)
model = nn.Sequential(nn.Flatten(),
                      nn.Linear(784, 512), nn.ReLU(), nn.Dropout(0.3),
                      nn.Linear(512, 512), nn.ReLU(), nn.Dropout(0.3),
                      nn.Linear(512, 10)).to(device)
opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=0.1)
h_combo = fit(model, train_loader, val_loader, loss_fn, opt, device, epochs=40, patience=8)

assert min(h_combo["val_loss"]) < min(h_base["val_loss"])
assert max(h_combo["val_acc"]) >= 0.81
print("Rätt!", round(min(h_combo["val_loss"]), 3), round(max(h_combo["val_acc"]), 3))

### Övning 3

In [ ]:
full_train, full_val, _ = fashion_mnist_loaders()
set_seed(42)
model = make_mlp().to(device)
h_full = fit(model, full_train, full_val, loss_fn,
             torch.optim.Adam(model.parameters(), lr=1e-3), device, epochs=10, verbose=False)
gap_full = h_full["train_acc"][-1] - h_full["val_acc"][-1]
gap_small = h_base["train_acc"][-1] - h_base["val_acc"][-1]
print(f"gap med 2 000 bilder:  {gap_small:.3f}  (val acc {h_base['val_acc'][-1]:.3f})")
print(f"gap med 50 000 bilder: {gap_full:.3f}  (val acc {h_full['val_acc'][-1]:.3f})")

**Slutsats:** med 25 gånger mer data blir gapet mycket mindre och val accuracy betydligt högre, utan någon regularisering alls. Mer (och varierad) data är den mest effektiva regulariseringen. Teknikerna ovan behövs mest när data är begränsad, och det är den nästan alltid. Data augmentation (L10) är ett sätt att "skapa" mer data.